# 서울시 생애주기별 사회적 고립 위험지역 분석

서울시 행정동 단위의 인구·생활이동·생활 인프라를 결합해 청년, 중장년, 노년의 상대적 고립 취약도를 계산하고 정책 선투입 후보를 찾는 분석이다.

이 공개본은 계산 구조와 검증된 집계 결과를 설명한다. 서울시 빅데이터캠퍼스 원본과 개인·통신 기반 상세 자료는 포함하지 않는다.

> 위험지수는 개인의 사회적 고립이나 고독사 확률이 아니다. 행정동 간 상대적 취약도를 비교하기 위한 규칙 기반 지수다.


## 공개 결과 요약

| 항목 | 결과 |
|---|---:|
| 기준 행정동 | 426개 |
| 청년 위험지수 유효 행 | 379개 |
| 중장년 위험지수 유효 행 | 379개 |
| 노년 위험지수 유효 행 | 378개 |
| 세 지수 완전 사례 | 378개 |
| 미분류 행정동 | 48개 |
| 전생애 중첩위험 | TOP9 |

70점 이상 행정동은 청년 40개, 중장년 52개, 노년 26개다. 네 군집은 완전 사례 378개만 사용했다.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# 원본은 공개 저장소에 포함하지 않는다.
PRIVATE_DATA = Path("data/private/integrated_isolation_table.csv")
print("원본 데이터 존재 여부:", PRIVATE_DATA.exists())


## 1. 분석 단위와 지표 방향

- 공간 단위: 서울시 행정동 코드 `ADM_CD`
- 청년: 20~39세
- 중장년: 40~64세
- 노년: 65세 이상
- 인구 축: 연령별 1인가구 집중도가 높을수록 위험 증가
- 이동 축: 생활이동이 적을수록 위험 증가
- 인프라 축: 의료·복지·교통 접근성이 낮을수록 위험 증가

자료마다 행정동 코드와 기준 시점이 달라 426개 기준 테이블을 유지한 채 결측을 보존했다. 점수·군집 결과에서 결측을 0점으로 바꾸지 않았다.


In [ ]:
RISK_COLUMNS = ["risk_youth", "risk_middle", "risk_elderly"]

WEIGHTS = {
    "youth": {"population": 0.20, "mobility": 0.40, "infrastructure": 0.40},
    "middle": {"population": 0.20, "mobility": 0.40, "infrastructure": 0.40},
    "elderly": {"population": 0.20, "mobility": 0.35, "infrastructure": 0.45},
}

INFRA_WEIGHTS = {
    "youth_middle": {"medical": 0.50, "transport": 0.50},
    "elderly": {"welfare": 0.60, "medical": 0.25, "transport": 0.15},
}

assert all(np.isclose(sum(w.values()), 1.0) for w in WEIGHTS.values())
assert all(np.isclose(sum(w.values()), 1.0) for w in INFRA_WEIGHTS.values())


## 2. 이상치 절삭과 정규화

시설 밀도처럼 일부 행정동의 값이 매우 큰 변수는 99백분위에서 상한을 두었다. 이후 0~100점으로 변환하고, 이동·인프라는 값이 낮을수록 위험점수가 높아지도록 방향을 뒤집었다.


In [ ]:
def clipped_minmax(series: pd.Series, *, reverse: bool = False) -> pd.Series:
    """99백분위 절삭 후 0~100점으로 변환한다. 결측은 유지한다."""
    values = pd.to_numeric(series, errors="coerce")
    upper = values.quantile(0.99)
    clipped = values.clip(upper=upper)
    low, high = clipped.min(), clipped.max()

    if pd.isna(low) or pd.isna(high) or np.isclose(low, high):
        score = pd.Series(np.nan, index=series.index, dtype=float)
    else:
        score = (clipped - low) / (high - low) * 100

    return 100 - score if reverse else score


## 3. 생애주기별 복합위험지수

청년과 중장년은 이동·인프라를 각각 40% 반영했다. 노년은 이동 35%, 인프라 45%로 조정해 근거리 돌봄·복지 접근성을 더 크게 반영했다.


In [ ]:
def build_lifecycle_risk(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()

    result["infra_youth_middle"] = (
        result["medical_risk"] * INFRA_WEIGHTS["youth_middle"]["medical"]
        + result["transport_risk"] * INFRA_WEIGHTS["youth_middle"]["transport"]
    )
    result["infra_elderly"] = (
        result["welfare_risk"] * INFRA_WEIGHTS["elderly"]["welfare"]
        + result["medical_risk"] * INFRA_WEIGHTS["elderly"]["medical"]
        + result["transport_risk"] * INFRA_WEIGHTS["elderly"]["transport"]
    )

    result["risk_youth"] = (
        result["youth_pop_n"] * 0.20
        + result["youth_move_n"] * 0.40
        + result["infra_youth_middle"] * 0.40
    )
    result["risk_middle"] = (
        result["middle_pop_n"] * 0.20
        + result["middle_move_n"] * 0.40
        + result["infra_youth_middle"] * 0.40
    )
    result["risk_elderly"] = (
        result["elderly_pop_n"] * 0.20
        + result["elderly_move_n"] * 0.35
        + result["infra_elderly"] * 0.45
    )
    return result


### 산출 결과

| 연령군 | 유효 행정동 | 평균 | 중앙값 | 70점 이상 | 최고 지역 |
|---|---:|---:|---:|---:|---|
| 청년 | 379 | 53.27 | 54.57 | 40 | 가리봉동 83.63 |
| 중장년 | 379 | 52.45 | 53.37 | 52 | 신원동 84.40 |
| 노년 | 378 | 50.22 | 50.00 | 26 | 가리봉동 84.52 |


## 4. 완전 사례만 사용한 군집 분석

결측을 평균이나 0으로 대체하면 지역의 실제 위험 구조를 왜곡할 수 있다. 세 위험지수와 군집 입력 변수가 모두 있는 행만 학습에 사용하고, 나머지는 `-1 / 미분류`로 보존한다.


In [ ]:
def assign_clusters(df: pd.DataFrame, feature_columns: list[str]) -> pd.DataFrame:
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler

    result = df.copy()
    complete = result[feature_columns].notna().all(axis=1)
    X = StandardScaler().fit_transform(result.loc[complete, feature_columns])

    model = KMeans(n_clusters=4, random_state=42, n_init=20)
    labels = model.fit_predict(X)

    result["cluster"] = -1
    result.loc[complete, "cluster"] = labels
    return result


CLUSTER_COUNTS = pd.Series(
    {
        "1인가구 집중형": 78,
        "도심상업형": 30,
        "활동적 도심형": 129,
        "생활안정형": 141,
        "미분류": 48,
    },
    name="행정동 수",
)
CLUSTER_COUNTS


## 5. 인프라 결핍 유형

의료·복지·교통 밀도가 각각 전체 행정동의 하위 30%인지 표시하고 결핍 축의 개수로 유형을 정했다. 이 기준은 운영 규칙이며 정책 효과로 최적화한 임계값이 아니다.


In [ ]:
def classify_deficit(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    density_columns = {
        "medical": "의료밀도",
        "welfare": "복지밀도",
        "transport": "교통밀도",
    }

    flag_columns = []
    for key, column in density_columns.items():
        threshold = result[column].quantile(0.30)
        flag = f"{key}_deficit_flag"
        result[flag] = result[column].lt(threshold).fillna(False).astype(int)
        flag_columns.append(flag)

    result["deficit_count"] = result[flag_columns].sum(axis=1)
    labels = {
        0: "0형: 인프라 양호형",
        1: "1형: 단일 결핍형",
        2: "2형: 복합 결핍형",
        3: "3형: 삼중 결핍형",
    }
    result["deficit_level"] = result["deficit_count"].map(labels)
    return result


DEFICIT_COUNTS = pd.Series(
    {"인프라 양호": 65, "단일 결핍": 155, "복합 결핍": 145, "삼중 결핍": 61},
    name="행정동 수",
)
DEFICIT_COUNTS


## 6. 전생애 중첩위험 TOP9

세 연령대별 위험지수 상위 20개 집합의 교집합을 구했다. 공통으로 포함된 9개 행정동의 평균 위험지수로 순위를 정했다.


In [ ]:
def lifecycle_top_intersection(df: pd.DataFrame, n: int = 20) -> pd.DataFrame:
    complete = df.dropna(subset=RISK_COLUMNS).copy()
    top_sets = [set(complete.nlargest(n, column)["ADM_CD"]) for column in RISK_COLUMNS]
    common_codes = set.intersection(*top_sets)

    result = complete[complete["ADM_CD"].isin(common_codes)].copy()
    result["mean_risk"] = result[RISK_COLUMNS].mean(axis=1)
    return result.sort_values("mean_risk", ascending=False)


TOP9 = pd.DataFrame(
    {
        "행정동": ["가리봉동", "신원동", "독산2동", "청림동", "신월3동", "망우3동", "창신2동", "신길4동", "청구동"],
        "평균위험지수": [84.1, 80.4, 80.3, 78.8, 78.7, 78.5, 78.1, 75.3, 74.7],
    },
    index=range(1, 10),
)
TOP9.index.name = "순위"
TOP9


## 7. 외부 취약지표와의 정합성

자치구별 평균 위험지수와 기초생활수급자 비율을 Spearman 순위상관으로 비교했다. 자살률과의 상관은 세 연령대 모두 통계적으로 유의하지 않았다. 따라서 외부 검증 결과는 생활취약성과 같은 방향을 보이는지 확인한 보조 근거로만 해석한다.


In [ ]:
def spearman_checks(gu_table: pd.DataFrame) -> pd.DataFrame:
    from scipy.stats import spearmanr

    rows = []
    for risk_column in RISK_COLUMNS:
        for external_column in ["수급자비율", "자살률_계"]:
            valid = gu_table[[risk_column, external_column]].dropna()
            rho, p_value = spearmanr(valid[risk_column], valid[external_column])
            rows.append(
                {
                    "risk": risk_column,
                    "external": external_column,
                    "n": len(valid),
                    "rho": rho,
                    "p_value": p_value,
                }
            )
    return pd.DataFrame(rows)


VALIDATION_RESULTS = pd.DataFrame(
    [
        ["청년", "기초생활수급자 비율", 25, 0.625, 0.001, "유의"],
        ["중장년", "기초생활수급자 비율", 25, 0.592, 0.002, "유의"],
        ["노년", "기초생활수급자 비율", 25, 0.413, 0.040, "유의"],
        ["청년", "자살률", 25, 0.274, 0.185, "비유의"],
        ["중장년", "자살률", 25, 0.246, 0.237, "비유의"],
        ["노년", "자살률", 25, 0.298, 0.148, "비유의"],
    ],
    columns=["연령군", "외부 지표", "n", "rho", "p_value", "판정"],
)
VALIDATION_RESULTS


## 8. 공개 범위와 재현 조건

- 공개: 점수 계산식, 결측 처리 원칙, 군집·TOP9·교차검증 코드 구조, 검증된 집계값
- 비공개: B031·B076 원본, 개인·통신 기반 상세 자료, 행정동별 전체 원천값
- 전체 재실행 조건: 원본 데이터 이용 권한, 행정동 코드 매핑표, GeoPandas·scikit-learn·SciPy 환경

### 해석상 한계

1. 1인가구·이동·인프라는 사회적 고립의 직접 측정값이 아니다.
2. 48개 행정동은 위험지수 결측으로 군집에서 제외됐다.
3. 일부 외부 검증은 25개 자치구 단위여서 행정동 결과와 공간 해상도가 다르다.
4. 가중치와 임계값은 정책 성과 데이터로 학습하거나 최적화하지 않았다.
5. 지역 단위 결과로 개인의 상태를 판단하면 안 된다.
